# Chapter 03. 텍스트를 숫자로 바꾸는 CountVectorizer와 TF-IDF

Chapter 02에서는 도서 제목에서 단어를 추출하고 빈도를 세었습니다. 이번 Chapter에서는 한 단계 더 나아가 **도서 제목을 머신러닝이 계산할 수 있는 숫자 벡터로 바꾸는 방법**을 학습합니다.

이번 Chapter의 핵심 흐름은 다음과 같습니다.

**전처리 파일 불러오기 → 제목 문자열 준비 → Bag of Words 이해 → CountVectorizer → 단어-문서 행렬 → 희소 행렬 → Count 상위 단어 → TF·DF·IDF 이해 → TfidfVectorizer → 도서별 주요 TF-IDF 단어 → 전체 평균 TF-IDF → Count와 TF-IDF 비교 → 원본 제목 검증 → fit/transform 구분 → 데이터 누수 주의 → 결과 저장 및 Markdown 정리**

이번에도 **해야 할 일 이해 → AI에게 질문 → 코드 초안 확인 → Notebook 실행 → 결과 확인 → 검증 → Markdown 정리** 순서로 진행합니다.

> 먼저 Chapter 01 Notebook을 끝까지 실행해 `notebooks/book-text-ml/book_bestseller_clean.csv`가 만들어져 있어야 합니다.

### 이번 Chapter의 핵심 질문

1. 왜 텍스트를 숫자로 바꿔야 하는가?
2. Bag of Words는 무엇인가?
3. CountVectorizer가 만드는 숫자는 무엇을 의미하는가?
4. 행, 열, 셀 값은 각각 무엇인가?
5. 같은 단어가 두 번 나오면 숫자는 어떻게 되는가?
6. 모든 문서에 자주 등장하는 단어는 정말 중요한가?
7. TF, DF, IDF는 각각 무엇인가?
8. TF-IDF 값이 크다는 것은 무엇을 의미하는가?
9. CountVectorizer와 TfidfVectorizer는 어떤 차이가 있는가?
10. 왜 모델 평가에서는 train 데이터에만 Vectorizer를 fit해야 하는가?

## 실습 1. Chapter 01 전처리 데이터 불러오기

### AI에게 질문

> Python과 pandas를 처음 배우고 있습니다.  
> Chapter 01에서 만든 `book_bestseller_clean.csv` 파일을 pandas로 `df_books`라는 DataFrame으로 불러오고 싶습니다.  
> 다음 내용을 확인하는 간단한 코드를 작성해 주세요.
>
> 1. 전체 행과 컬럼 개수  
> 2. 컬럼 이름  
> 3. 상품명 앞의 10개  
> 4. 상품명 결측치 개수  
>
> CSV는 `utf-8-sig` 인코딩으로 저장했습니다.  
> 초보자가 이해하기 쉽도록 작성해 주세요.

### AI 답변

`pd.read_csv()`로 CSV를 읽고, `shape`, `columns`, `isna().sum()`, `head()`를 이용해 데이터가 정상적으로 연결되었는지 확인하면 됩니다. 처음부터 벡터화를 하지 않고 **입력 데이터가 제대로 들어왔는지 먼저 확인**하는 것이 중요합니다.

### 초보자 상세 설명

이 실습은 뒤의 모든 분석이 의존하는 입력 데이터를 검증하는 단계입니다. shape로 행·열 수를 확인하고, columns로 실제 컬럼 이름을 확인하며, 상품명 결측치와 한글 표시 상태까지 눈으로 확인합니다. 파일이 열렸다는 사실만으로 끝내지 말고 Chapter 01에서 만든 결과와 행 수가 크게 달라지지 않았는지도 비교해야 합니다.

In [ ]:
# pandas를 불러옵니다.
import pandas as pd

# Notebook에서 DataFrame과 Markdown을 명확하게 출력할 때 사용합니다.
from IPython.display import display, Markdown

# Chapter 01에서 만든 전처리 CSV 파일 경로입니다.
# 현재 Notebook은 프로젝트 루트에서 실행하므로 전체 프로젝트 기준 경로를 사용합니다.
DATA_PATH = "notebooks/book-text-ml/book_bestseller_clean.csv"

# CSV 파일을 df_books라는 DataFrame으로 불러옵니다.
df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

# 전체 행과 컬럼 개수를 확인합니다.
print("데이터 크기:", df_books.shape)

# 컬럼 이름을 확인합니다.
print("컬럼:", df_books.columns.tolist())

# 상품명 결측치 개수를 확인합니다.
print("상품명 결측치:", df_books["상품명"].isna().sum())

# 상품명 앞의 10개를 확인합니다.
df_books[["상품명"]].head(10)

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 파일과 핵심 컬럼을 한 번 더 확인합니다.
from pathlib import Path

print('CSV 파일 존재:', Path(DATA_PATH).exists())
print("'상품명' 컬럼 존재:", '상품명' in df_books.columns)
print('첫 번째 상품명:', df_books['상품명'].iloc[0])

### 실습 1 결과 확인 및 정리

파일이 정상적으로 열렸는지, `상품명` 컬럼이 실제로 존재하는지, 한글이 깨지지 않는지 확인합니다. Chapter 01 결과와 행 개수가 크게 달라졌다면 파일을 잘못 읽은 것은 아닌지도 확인해야 합니다.

## 실습 2. 분석할 제목 문자열 준비하기

Vectorizer에 전달할 제목을 문자열 형태로 정리합니다. 결측치는 빈 문자열로 바꾸고, 문자열로 변환한 뒤 앞뒤 공백을 제거합니다. 완전히 빈 제목은 제외하고 인덱스를 다시 정리합니다.

### 초보자 상세 설명

Vectorizer에 넣기 전에 제목을 일정한 문자열 형태로 정리합니다. fillna("")는 결측치를 빈 문자열로 바꾸고, astype(str)은 문자열로 통일하며, str.strip()은 앞뒤 공백을 제거합니다. 마지막으로 빈 제목을 제외하고 reset_index(drop=True)로 0부터 다시 번호를 매깁니다. 이렇게 해야 iloc으로 특정 제목을 확인할 때 행 번호가 깔끔하게 이어집니다.

In [ ]:
# 상품명 결측치를 빈 문자열로 바꾸고 문자열로 변환한 뒤 앞뒤 공백을 제거합니다.
titles = (
    df_books["상품명"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# 완전히 비어 있는 제목은 제외하고 인덱스를 0부터 다시 정리합니다.
titles = titles[titles != ""].reset_index(drop=True)

print("사용할 제목 수:", len(titles))
titles.head(10)

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 정리 전/후 제목 개수를 비교합니다.
raw_title_count = len(df_books['상품명'])
clean_title_count = len(titles)

print('원본 행 수:', raw_title_count)
print('정리 후 사용할 제목 수:', clean_title_count)
print('제외된 행 수:', raw_title_count - clean_title_count)
print('빈 제목 남아 있음:', (titles == '').any())

### 실습 2 결과 확인 및 정리

`titles`는 이번 Chapter에서 실제로 Vectorizer에 전달할 제목 목록입니다. 이번에는 Chapter 02의 Kiwi 결과를 바로 쓰지 않고 **원래 도서 제목 문자열을 그대로 사용해 Vectorizer의 기본 원리부터 이해**합니다.

## 실습 3. 머신러닝은 왜 텍스트를 숫자로 바꿀까?

머신러닝 알고리즘은 일반적으로 숫자를 입력받아 계산합니다. 문자열인 도서 제목을 그대로 더하거나 곱해서 학습할 수 없기 때문에, 텍스트를 **숫자 벡터**로 바꾸는 과정이 필요합니다.

쉽게 말하면 다음과 같습니다.

**도서 제목 → 사용된 단어 확인 → 단어마다 열 생성 → 등장 횟수나 가중치를 숫자로 기록 → 숫자 벡터**

이 과정을 **텍스트 벡터화(Vectorization)**라고 합니다.

### 초보자 상세 설명

머신러닝은 문자열 그 자체를 계산하기보다 숫자 feature를 입력으로 받습니다. 따라서 도서 제목을 단어 단위의 열로 만들고, 각 문서가 그 단어를 얼마나 가지고 있는지를 숫자로 표현합니다. 중요한 것은 숫자 하나만 보는 것이 아니라 그 숫자가 어떤 단어 열에 해당하는지 함께 해석하는 것입니다.

### 실습 3 결과 확인 및 정리

문자열 자체를 모델이 직접 계산하는 것이 아니라, 각 문장을 일정한 기준의 숫자 배열로 바꾼 뒤 모델에 입력한다는 점이 핵심입니다.

## 실습 4. 아주 작은 예제로 먼저 이해하기

실제 도서 제목 전체를 바로 사용하면 행과 열이 많아져 원리를 보기 어렵습니다. 먼저 세 문장만 사용합니다.

### 초보자 상세 설명

이 실습은 CountVectorizer를 이해하기 위한 핵심 준비 단계입니다. sample_docs를 만든 뒤 바로 다음 실습으로 넘어가지 말고, 사람이 직접 서로 다른 단어를 모아 feature 순서를 정하고 각 문장을 벡터로 바꿔 봐야 합니다. 예를 들어 feature 순서를 [데이터, 머신러닝, 분석, 입문, 파이썬]으로 잡으면 '파이썬 데이터 분석'은 [1, 0, 1, 0, 1]이 됩니다. 이 숫자는 중요도 점수가 아니라 등장 횟수입니다.

In [ ]:
# 원리를 보기 위한 작은 예제 문서 3개입니다.
sample_docs = [
    "파이썬 데이터 분석",
    "파이썬 머신러닝",
    "데이터 분석 입문",
]

sample_docs

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 사람이 직접 feature 순서를 정해 봅니다.
manual_features = ['데이터', '머신러닝', '분석', '입문', '파이썬']

print('feature 순서:', manual_features)

# 각 문장에서 feature가 몇 번 등장하는지 직접 세어 벡터를 만듭니다.
for doc in sample_docs:
    vector = [doc.split().count(feature) for feature in manual_features]
    print(doc, '->', vector)

# 첫 번째 문장은 [1, 0, 1, 0, 1]이 되는지 직접 확인합니다.
first_manual_vector = [sample_docs[0].split().count(feature) for feature in manual_features]
print('첫 번째 문장 벡터:', first_manual_vector)

### 실습 4 결과 확인 및 정리

세 문장에서 사용된 핵심 단어를 모으면 파이썬, 데이터, 분석, 머신러닝, 입문처럼 정리할 수 있습니다. 이 단어들을 열로 놓고 각 문장에서 몇 번 등장했는지를 숫자로 기록하면 문장이 벡터가 됩니다.

## 실습 5. Bag of Words 이해하기

### AI에게 질문

> Python과 머신러닝을 처음 배우고 있습니다.  
> CountVectorizer를 이용해 텍스트를 숫자로 바꾸는 과정을 배우고 있습니다.  
> 다음 세 문장을 예로 사용해서 Bag of Words와 단어-문서 행렬을 초보자가 이해할 수 있게 설명해 주세요.
>
> - 파이썬 데이터 분석
> - 파이썬 머신러닝
> - 데이터 분석 입문
>
> 행, 열, 셀 값이 각각 무엇을 의미하는지도 설명해 주세요.  
> 수식보다 직관적인 설명을 우선해 주세요.

### AI 답변

Bag of Words는 문장을 **단어가 담긴 주머니**처럼 보고, 단어의 순서보다는 어떤 단어가 몇 번 등장했는지에 초점을 맞추는 방법입니다.

예를 들어 열이 `[데이터, 머신러닝, 분석, 입문, 파이썬]` 순서라면 `파이썬 데이터 분석`은 `[1, 0, 1, 0, 1]`처럼 표현할 수 있습니다.

- **행(row)** = 문서 또는 도서 제목
- **열(column)** = 단어
- **셀 값(value)** = 그 문서에서 해당 단어가 등장한 횟수

따라서 `파이썬 데이터 분석`과 `데이터 파이썬 분석`처럼 단어 순서만 다르고 등장 횟수가 같다면 기본 Bag of Words에서는 같은 벡터가 될 수 있습니다.

### 초보자 상세 설명

Bag of Words는 단어 순서보다 어떤 단어가 몇 번 등장하는지에 집중합니다. 그래서 '파이썬 데이터 분석'과 '데이터 파이썬 분석'은 어순이 다르더라도 같은 단어가 같은 횟수로 나오면 같은 Count 벡터가 될 수 있습니다. 단순하고 해석하기 쉬운 대신 문맥과 어순을 충분히 표현하지 못한다는 한계가 있습니다.

### 실습 5 결과 확인 및 정리

Bag of Words는 간단하고 직관적이지만 **단어 순서와 깊은 문맥을 충분히 표현하지 못한다**는 한계가 있습니다. 이번 과정에서는 먼저 이 단순한 구조를 이해한 뒤 분류와 추천에 연결합니다.

## 실습 6. CountVectorizer 설치 확인하기

CountVectorizer는 `scikit-learn`에 포함되어 있습니다. 설치되어 있지 않다면 VS Code 터미널에서 다음 명령을 실행합니다.

```powershell
python -m pip install scikit-learn
```

Notebook에서 아래 import가 오류 없이 실행되면 사용할 준비가 된 것입니다.

### 초보자 상세 설명

CountVectorizer는 scikit-learn에 포함되어 있습니다. 여기서 중요한 것은 패키지를 설치한 Python과 Notebook 커널의 Python이 같은지 확인하는 것입니다. 터미널에서 설치가 성공했어도 Notebook이 다른 Python을 쓰고 있으면 ModuleNotFoundError가 날 수 있습니다. sys.executable과 sys.version을 확인하면 현재 Notebook 환경을 정확히 알 수 있습니다.

### 현재 Notebook이 사용하는 Python부터 확인하기

패키지 설치가 성공했는데도 import가 안 되는 가장 흔한 이유는 **터미널의 Python과 Notebook 커널의 Python이 다른 경우**입니다. 먼저 현재 Notebook이 어떤 Python을 사용하는지 확인합니다.

In [ ]:
import sys

print('Python 실행 파일:')
print(sys.executable)

print('\nPython 버전:')
print(sys.version)

try:
    import sklearn
    print('\nscikit-learn 버전:', sklearn.__version__)
except ModuleNotFoundError:
    print('\n현재 Notebook Python에는 scikit-learn이 설치되어 있지 않습니다.')
    print('현재 커널에 설치하려면 Notebook 셀에서 다음 명령을 실행합니다:')
    print(f'"{sys.executable}" -m pip install scikit-learn')

In [ ]:
# CountVectorizer를 불러옵니다.
from sklearn.feature_extraction.text import CountVectorizer

print("CountVectorizer import 성공")

### 실습 6 결과 확인 및 정리

만약 `ModuleNotFoundError: No module named 'sklearn'`이 나오면 패키지가 현재 Notebook 커널과 같은 Python 환경에 설치되어 있는지 확인합니다.

## 실습 7. 작은 예제에 CountVectorizer 적용하기

`CountVectorizer()`를 만든 뒤 `fit_transform()`을 사용합니다.

- `fit` → 문서에서 어떤 단어를 열로 사용할지 학습
- `transform` → 학습한 기준으로 각 문서를 숫자 벡터로 변환
- `fit_transform` → 두 작업을 한 번에 수행

### 초보자 상세 설명

fit_transform()은 두 작업을 한 번에 합니다. fit은 문서에서 어떤 단어를 열로 사용할지 학습하고, transform은 그 기준으로 각 문서를 숫자 벡터로 바꿉니다. 뒤의 train/test 실습에서는 이 둘을 구분해야 데이터 누수를 피할 수 있으므로 지금부터 의미를 나눠 이해하는 것이 중요합니다.

In [ ]:
# CountVectorizer 객체를 만듭니다.
count_vectorizer_sample = CountVectorizer()

# sample_docs에서 단어 사전을 학습하고 동시에 숫자 행렬로 변환합니다.
X_count_sample = count_vectorizer_sample.fit_transform(sample_docs)

print("작은 예제 행렬 크기:", X_count_sample.shape)

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# fit과 transform을 따로 실행해 같은 흐름인지 확인합니다.
count_vectorizer_step = CountVectorizer()

# 1) 단어 사전을 학습합니다.
count_vectorizer_step.fit(sample_docs)

# 2) 학습한 기준으로 숫자 행렬로 바꿉니다.
X_count_step = count_vectorizer_step.transform(sample_docs)

print('fit 후 feature:', count_vectorizer_step.get_feature_names_out())
print('transform 결과 shape:', X_count_step.shape)
print('fit+transform 결과가 fit_transform과 같은가?:', (X_count_step != X_count_sample).nnz == 0)

### 실습 7 결과 확인 및 정리

`fit_transform()`은 단어 사전을 만들고, 각 문서를 그 사전 기준의 숫자 벡터로 바꾸는 두 단계를 한 번에 수행합니다.

## 실습 8. 생성된 단어 사전 확인하기

Vectorizer가 어떤 단어를 열로 만들었는지 `get_feature_names_out()`으로 확인합니다.

### 초보자 상세 설명

get_feature_names_out()은 행렬의 열 이름을 알려 줍니다. 숫자 벡터의 첫 번째 값이 어떤 단어를 의미하는지는 feature 순서를 알아야 해석할 수 있습니다. Vectorizer가 정한 순서를 기준으로 보며 사람이 임의로 예상한 순서와 같다고 가정하지 않습니다.

In [ ]:
# Vectorizer가 만든 feature(단어) 목록을 가져옵니다.
feature_names = count_vectorizer_sample.get_feature_names_out()

print(feature_names)

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# feature가 몇 번째 열에 배치되었는지 표로 확인합니다.
feature_index_table = pd.DataFrame({
    '열번호': range(len(feature_names)),
    '단어': feature_names,
})

feature_index_table

### 실습 8 결과 확인 및 정리

숫자 벡터만 보면 각 숫자가 어떤 단어를 뜻하는지 알 수 없습니다. 따라서 벡터를 해석할 때는 **feature 이름의 순서와 함께 확인**해야 합니다.

## 실습 9. 단어-문서 행렬 확인하기

작은 예제이므로 희소 행렬을 `toarray()`로 바꿔 전체 값을 확인해도 괜찮습니다. DataFrame으로 만들면 행과 열의 의미를 더 쉽게 읽을 수 있습니다.

### 초보자 상세 설명

작은 예제에서 행렬을 DataFrame으로 바꾸면 행=문서, 열=단어, 값=등장 횟수라는 구조가 눈에 보입니다. 첫 번째 행을 직접 읽어 원문에 있는 단어는 1, 없는 단어는 0인지 확인해 보는 것이 CountVectorizer 이해의 핵심입니다.

In [ ]:
# 작은 예제이므로 Dense 배열로 바꿔 확인합니다.
sample_count_df = pd.DataFrame(
    X_count_sample.toarray(),
    columns=feature_names,
    index=sample_docs,
)

sample_count_df

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 첫 번째 문장의 feature 순서와 숫자 벡터를 직접 비교합니다.
print('feature 순서:')
print(feature_names)

print('\n첫 번째 문장:')
print(sample_docs[0])

print('\n첫 번째 문장의 숫자 벡터:')
print(X_count_sample.toarray()[0])

### 실습 9 결과 확인 및 정리

표에서 **행은 문서, 열은 단어, 값은 등장 횟수**입니다. 예를 들어 `파이썬 데이터 분석` 행에서 파이썬·데이터·분석은 1, 등장하지 않은 단어는 0인지 직접 확인합니다.

## 실습 10. 같은 단어가 여러 번 나오면 어떻게 될까?

CountVectorizer는 기본적으로 단어의 존재 여부만 기록하는 것이 아니라 **등장 횟수**를 기록합니다.

### 초보자 상세 설명

CountVectorizer의 기본 값은 단순 존재 여부가 아니라 등장 횟수입니다. 한 문장에 '파이썬'이 두 번 나오면 해당 열 값은 2가 됩니다. 따라서 Count 숫자를 중요도 점수로 오해하지 않고 우선 빈도라고 이해합니다.

In [ ]:
# 같은 단어가 반복되는 예제입니다.
repeat_docs = [
    "파이썬 파이썬 데이터",
    "데이터 분석",
]

repeat_vectorizer = CountVectorizer()
X_repeat = repeat_vectorizer.fit_transform(repeat_docs)

repeat_df = pd.DataFrame(
    X_repeat.toarray(),
    columns=repeat_vectorizer.get_feature_names_out(),
    index=repeat_docs,
)

repeat_df

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# '파이썬' 열을 찾아 실제 값이 2인지 확인합니다.
repeat_terms = repeat_vectorizer.get_feature_names_out()
python_col = list(repeat_terms).index('파이썬')

print('feature 순서:', repeat_terms)
print("'파이썬' 열 번호:", python_col)
print("첫 번째 문장의 '파이썬' Count:", X_repeat.toarray()[0, python_col])

### 실습 10 결과 확인 및 정리

첫 번째 문장에서 `파이썬`이 두 번 등장했다면 해당 셀 값은 **2**가 됩니다. 이것이 CountVectorizer의 기본 Count 의미입니다.

## 실습 11. CountVectorizer의 기본 토큰 기준 확인하기

CountVectorizer는 기본 설정에서 자체 토큰 패턴을 사용합니다. 기본 토큰 기준에서는 한 글자 토큰이 제외될 수 있으므로 직접 확인합니다.

### 초보자 상세 설명

CountVectorizer는 내부 token_pattern을 사용합니다. 기본 설정에서는 일반적으로 두 글자 이상의 단어 문자를 대상으로 하기 때문에 한 글자 영문 R 같은 것은 빠질 수 있습니다. 원문에 글자가 있다고 해서 반드시 feature가 되는 것은 아니며 토큰화 규칙이 결과에 영향을 줍니다.

In [ ]:
# 한 글자 영문 토큰 R이 포함된 예제입니다.
test_docs = ["AI 데이터 분석 R 파이썬"]

test_vectorizer = CountVectorizer()
X_test = test_vectorizer.fit_transform(test_docs)

print(test_vectorizer.get_feature_names_out())

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# CountVectorizer의 기본 token_pattern을 확인합니다.
print('기본 token_pattern:', test_vectorizer.token_pattern)
print('원문:', test_docs[0])
print('생성된 feature:', test_vectorizer.get_feature_names_out())

### 실습 11 결과 확인 및 정리

Chapter 02에서 최소 글자수를 정했던 것처럼 Vectorizer도 **어떤 문자열을 하나의 단어로 볼지**에 따라 결과가 달라집니다. 기본 옵션을 무조건 정답으로 생각하지 않습니다.

## 실습 12. 실제 도서 제목에 CountVectorizer 적용하기

이제 실제 `titles` 전체에 CountVectorizer를 적용합니다. 여기서는 Vectorizer 자체를 이해하기 위한 탐색 단계이므로 전체 제목에 `fit_transform()`을 사용합니다.

### 초보자 상세 설명

실제 titles 전체에 적용한 뒤 X_count.shape의 행 수가 len(titles)와 같은지, 열 수가 len(count_terms)와 같은지 직접 확인합니다. 이 두 관계가 맞아야 문서 수와 feature 수가 논리적으로 연결된 것입니다.

In [ ]:
# 실제 도서 제목용 CountVectorizer를 만듭니다.
count_vectorizer = CountVectorizer()

# 전체 제목에서 단어 사전을 만들고 Count 행렬로 변환합니다.
X_count = count_vectorizer.fit_transform(titles)

# 열에 해당하는 단어 목록을 가져옵니다.
count_terms = count_vectorizer.get_feature_names_out()

print("문서 수:", X_count.shape[0])
print("단어 수:", X_count.shape[1])
print("행렬 크기:", X_count.shape)

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 행 수와 제목 수, 열 수와 feature 수가 맞는지 검증합니다.
print('제목 수 == 행 수:', len(titles) == X_count.shape[0])
print('feature 수 == 열 수:', len(count_terms) == X_count.shape[1])
print('처음 20개 feature:')
print(count_terms[:20])

### 실습 12 결과 확인 및 정리

`X_count.shape[0]`은 사용한 제목 수이고 `X_count.shape[1]`은 CountVectorizer가 만든 feature 수입니다. 실제 숫자는 실행 결과를 기준으로 확인합니다.

## 실습 13. vocabulary_ 확인하기

`vocabulary_`에는 단어와 열 번호의 대응 정보가 들어 있습니다.

### 초보자 상세 설명

vocabulary_의 숫자는 빈도가 아니라 열 번호입니다. 예를 들어 ('파이썬', 123)은 파이썬이 123번 등장했다는 뜻이 아니라 행렬의 123번 열에 배치되었다는 뜻입니다. 실제 등장 횟수는 행렬 셀 값에서 확인합니다.

In [ ]:
# 단어와 열 번호의 대응 일부를 확인합니다.
list(count_vectorizer.vocabulary_.items())[:20]

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# vocabulary_의 단어 하나를 골라 열 번호와 feature 이름을 비교합니다.
sample_term, sample_col = next(iter(count_vectorizer.vocabulary_.items()))

print('예시 단어:', sample_term)
print('vocabulary_ 열 번호:', sample_col)
print('그 열 번호의 feature 이름:', count_terms[sample_col])

### 실습 13 결과 확인 및 정리

`vocabulary_`의 숫자는 **빈도수가 아니라 행렬에서 그 단어가 위치한 열 번호**입니다. 행렬의 셀 값이 실제 등장 횟수라는 점과 구분해야 합니다.

## 실습 14. 희소 행렬(Sparse Matrix) 이해하기

실제 도서 제목은 짧은데 전체 단어 사전은 매우 큽니다. 따라서 한 제목에서 대부분의 단어는 등장하지 않아 0이 됩니다.

이처럼 0이 매우 많은 행렬을 **희소 행렬(Sparse Matrix)**이라고 합니다. scikit-learn의 Vectorizer는 메모리를 효율적으로 사용하기 위해 기본적으로 희소 행렬을 반환합니다.

### 초보자 상세 설명

실제 텍스트 행렬은 대부분 0입니다. 제목 하나에는 전체 단어 사전 중 일부만 등장하기 때문입니다. 희소 행렬은 0을 전부 저장하는 대신 값이 있는 위치를 중심으로 저장해 메모리를 아낍니다. 그래서 전체 X_count를 무조건 toarray()로 바꾸지 않습니다.

In [ ]:
# 실제 Count 행렬의 자료형을 확인합니다.
print(type(X_count))

# 작은 예제는 Dense 배열로 보아도 괜찮습니다.
print(X_count_sample.toarray())

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 실제 Count 행렬의 희소성을 계산합니다.
count_total_cells = X_count.shape[0] * X_count.shape[1]
count_non_zero = X_count.nnz
count_zero_ratio = 1 - (count_non_zero / count_total_cells)

print('전체 셀 수:', count_total_cells)
print('0이 아닌 셀 수:', count_non_zero)
print('0의 비율:', round(count_zero_ratio, 4))

### 실습 14 결과 확인 및 정리

작은 예제에서는 `toarray()`가 괜찮지만 실제 전체 행렬을 무조건 `X_count.toarray()`로 바꾸면 메모리를 많이 사용할 수 있습니다. 필요한 행이나 집계값만 확인하는 습관이 중요합니다.

## 실습 15. 실제 데이터의 첫 번째 제목 벡터 확인하기

전체 행렬을 Dense로 바꾸지 않고 첫 번째 행에서 0이 아닌 값만 가져와 원래 제목과 비교합니다.

### 초보자 상세 설명

전체 행렬을 펼치지 않고 첫 번째 행의 0이 아닌 값만 확인합니다. indices는 값이 존재하는 열 번호, data는 실제 Count 값입니다. 열 번호를 count_terms와 연결하면 원래 제목이 어떤 token과 빈도로 바뀌었는지 직접 검증할 수 있습니다.

In [ ]:
# 첫 번째 제목을 확인합니다.
print("첫 번째 제목:", titles.iloc[0])

# 첫 번째 행만 가져옵니다.
first_row = X_count.getrow(0)

# 0이 아닌 열 번호와 실제 Count 값을 가져옵니다.
indices = first_row.indices
values = first_row.data

# 열 번호를 실제 단어 이름으로 바꿔 봅니다.
first_title_terms = [
    (count_terms[index], value)
    for index, value in zip(indices, values)
]

first_title_terms

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 첫 번째 제목과 실제 token/Count를 표로 확인합니다.
first_title_df = pd.DataFrame(first_title_terms, columns=['단어', 'Count'])
first_title_df = first_title_df.sort_values('Count', ascending=False)

print('원래 제목:')
print(titles.iloc[0])
display(first_title_df)

### 실습 15 결과 확인 및 정리

출력된 단어가 원래 제목에 실제로 존재하는지, 등장 횟수가 맞는지, 숫자와 영문은 어떻게 처리되었는지 직접 확인합니다. Vectorizer 결과를 한 행이라도 직접 읽어 보는 것이 중요합니다.

## 실습 16. 전체 데이터에서 많이 등장한 단어 확인하기

Count 행렬의 각 열을 합하면 모든 제목을 합쳐 각 단어가 총 몇 번 등장했는지 계산할 수 있습니다.

### 초보자 상세 설명

X_count.sum(axis=0)은 각 단어 열을 모든 문서에 걸쳐 합칩니다. 따라서 전체 제목에서 각 단어가 총 몇 번 등장했는지 계산할 수 있습니다. ravel()은 결과를 1차원 배열로 펴서 DataFrame에 넣기 쉽게 만드는 과정입니다.

In [ ]:
# 배열 계산을 위해 numpy를 불러옵니다.
import numpy as np

# 각 열의 합 = 전체 문서에서 해당 단어가 등장한 총 횟수
count_sums = np.asarray(X_count.sum(axis=0)).ravel()

# 단어와 전체 등장 횟수를 DataFrame으로 만듭니다.
count_summary = pd.DataFrame({
    "단어": count_terms,
    "전체등장횟수": count_sums,
})

# 등장 횟수가 많은 순서로 정렬합니다.
count_summary = count_summary.sort_values(
    "전체등장횟수",
    ascending=False,
).reset_index(drop=True)

count_summary.head(30)

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 상위 10개를 확인하고 전체 합도 검증합니다.
display(count_summary.head(10))

print(
    '요약표 총합 == 행렬 총합:',
    int(count_summary['전체등장횟수'].sum()) == int(X_count.sum())
)

### 실습 16 결과 확인 및 정리

Chapter 02의 상위 단어와 완전히 같지 않을 수 있습니다. Chapter 02는 Kiwi, 품사 필터, 최소 글자수, 불용어를 사용했고 현재 CountVectorizer는 자체 기본 토큰 기준을 사용하기 때문입니다. 결과가 다르면 **전처리 조건을 먼저 비교**합니다.

## 실습 17. Count 상위 단어 저장하기

Count 기준 상위 30개 단어를 CSV 파일로 저장합니다.

### 초보자 상세 설명

CSV 저장 후에는 파일이 존재하는지, 저장된 행 수가 30개인지, 다시 읽었을 때 한글과 값이 정상인지 확인합니다. 저장 명령이 실행되었다는 것과 실제 결과 파일이 올바르게 만들어졌다는 것은 별도로 검증해야 합니다.

In [ ]:
# 상위 30개만 복사합니다.
count_top30 = count_summary.head(30).copy()

# 결과 파일 경로입니다.
COUNT_TOP_PATH = "notebooks/book-text-ml/chapter03_count_top_terms.csv"

# Excel에서도 한글이 잘 보이도록 utf-8-sig로 저장합니다.
count_top30.to_csv(
    COUNT_TOP_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료:", COUNT_TOP_PATH)

# 저장한 파일을 다시 읽어 앞부분을 확인합니다.
pd.read_csv(COUNT_TOP_PATH, encoding="utf-8-sig").head()

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 저장된 CSV가 실제로 존재하고 30행인지 확인합니다.
count_path = Path(COUNT_TOP_PATH)
print('파일 존재:', count_path.exists())

count_saved = pd.read_csv(COUNT_TOP_PATH, encoding='utf-8-sig')
print('저장된 행 수:', len(count_saved))
display(count_saved.head())

### 실습 17 결과 확인 및 정리

최종 제출 결과물 중 하나인 `chapter03_count_top_terms.csv`를 만들었습니다. 저장 후 다시 읽어 보는 것은 파일이 실제로 정상 생성되었는지 확인하는 간단한 검증입니다.

## 실습 18. Count 방식의 한계 생각해 보기

CountVectorizer는 이해하기 쉽지만 **전체 문서에서 흔한 단어와 특정 문서에서 특징적인 단어를 구분하지 않습니다.**

예를 들어 거의 모든 제목에서 반복되는 일반적인 단어는 전체 Count가 매우 클 수 있습니다. 하지만 그 단어가 특정 문서를 다른 문서와 구분하는 데 항상 유용한 것은 아닙니다.

반대로 특정 문서에서만 상대적으로 두드러지는 단어는 전체 Count가 작아도 그 문서의 특징을 더 잘 나타낼 수 있습니다. 이 생각에서 TF-IDF가 출발합니다.

### 초보자 상세 설명

Count는 많이 등장한 단어를 찾는 데 직관적이지만 거의 모든 문서에 공통으로 나타나는 단어도 높은 값을 가질 수 있습니다. 특정 문서를 구분하는 특징을 보고 싶다면 전체 문서에서 얼마나 흔한지까지 고려할 필요가 있고, 여기서 TF-IDF로 연결됩니다.

### 실습 18 결과 확인 및 정리

Count가 높다는 것은 **많이 등장했다**는 뜻이지, 특정 문서를 구별하는 데 반드시 더 중요한 단어라는 뜻은 아닙니다.

## 실습 19. TF 이해하기

TF는 **Term Frequency**의 약자입니다. 초보자 단계에서는 “한 문서 안에서 특정 단어가 얼마나 나타나는가?”라고 이해하면 됩니다.

예를 들어 `파이썬 파이썬 데이터 분석`에서는 파이썬이 두 번, 데이터와 분석이 한 번 등장합니다.

### 초보자 상세 설명

TF는 한 문서 안에서 특정 단어가 얼마나 나타나는지를 보는 관점입니다. 다만 scikit-learn의 최종 TF-IDF 값은 정규화와 IDF가 함께 적용되므로 최종 숫자를 단순 등장 횟수로 읽으면 안 됩니다.

### 실습 19 결과 확인 및 정리

scikit-learn의 최종 TF-IDF 값에는 정규화와 IDF가 함께 반영되므로 최종 숫자를 단순한 등장 횟수 자체로 해석하면 안 됩니다.

## 실습 20. DF 이해하기

DF는 **Document Frequency**입니다. “특정 단어가 전체 문서 중 몇 개 문서에 등장하는가?”를 의미합니다.

예를 들어 제목이 100개이고 `데이터`가 80개 제목에 등장하면 DF가 높습니다. 반대로 어떤 전문용어가 2개 제목에만 등장하면 DF가 낮습니다.

### 초보자 상세 설명

DF는 특정 단어가 몇 개 문서에 등장하는지를 뜻합니다. 한 문서에서 같은 단어가 여러 번 반복되어도 그 문서 자체는 하나이므로 DF는 문서 수 기준입니다. 전체 Count와 DF는 서로 다른 정보를 보여 줍니다.

### 실습 20 결과 확인 및 정리

DF는 단어가 전체 문서 집합에 얼마나 널리 퍼져 있는지를 보여 줍니다.

## 실습 21. IDF 이해하기

IDF는 **Inverse Document Frequency**입니다.

- 거의 모든 문서에 등장하는 단어 → 문서를 구분하는 힘이 상대적으로 작을 수 있음 → IDF가 상대적으로 작아짐
- 적은 문서에 등장하는 단어 → 특정 문서를 구분하는 데 도움이 될 수 있음 → IDF가 상대적으로 커질 수 있음

scikit-learn은 기본적으로 smoothing이 포함된 IDF 계산을 사용합니다. 이번 단계에서는 공식을 외우기보다 **전체에서 흔한 단어의 가중치를 낮추고 상대적으로 드문 단어의 가중치를 높이는 역할**로 이해합니다.

### 초보자 상세 설명

IDF는 전체 문서에 너무 널리 퍼진 단어의 가중치를 상대적으로 낮추고, 일부 문서에서만 보이는 단어의 가중치를 상대적으로 높이는 역할을 합니다. 드문 단어가 현실에서 무조건 중요하다는 뜻은 아니며 현재 문서 집합에서 구분력 있는 특징일 가능성을 반영합니다.

### 실습 21 결과 확인 및 정리

IDF의 절대 숫자를 외우는 것보다 어떤 단어가 여러 문서에 널리 퍼져 있을수록 IDF가 상대적으로 어떻게 변하는지 이해하는 것이 중요합니다.

## 실습 22. TF-IDF를 한 문장으로 정리하기

### AI에게 질문

> CountVectorizer까지 이해한 초보자입니다.  
> TF, DF, IDF, TF-IDF를 서로 연결해서 설명해 주세요.  
> 특히 '모든 문서에 자주 나오는 단어'와 '특정 문서에서만 상대적으로 눈에 띄는 단어'가 왜 다른 가중치를 가질 수 있는지 작은 예제로 설명해 주세요.  
> 수식을 외우기보다 의미를 이해할 수 있게 설명해 주세요.

### AI 답변

TF는 **이 문서 안에서 얼마나 자주 나오는가**, DF는 **전체 문서 중 몇 개 문서에 나오는가**, IDF는 **전체 문서에서 너무 흔한 단어의 영향은 줄이고 상대적으로 드문 단어의 영향은 높이는 관점**입니다.

TF-IDF는 이 두 관점을 결합해 **이 문서에서는 눈에 띄지만 전체 문서에서는 너무 흔하지 않은 단어**에 상대적으로 높은 값을 줄 수 있습니다.

예를 들어 모든 도서 제목에 `책`이라는 단어가 들어 있고 한 제목에만 `약동학`이 있다면, 단순 Count에서는 둘 다 등장 횟수가 1일 수 있습니다. 하지만 `책`은 거의 모든 문서에서 흔하고 `약동학`은 특정 문서에서만 나타나므로 TF-IDF에서는 둘의 가중치가 달라질 수 있습니다.

### 초보자 상세 설명

TF-IDF는 이 문서 안에서 얼마나 나타나는가(TF)와 전체 문서에서는 얼마나 흔한가(IDF)를 함께 봅니다. 값이 높다는 것은 현재 문서 집합과 현재 설정에서 상대적으로 두드러진 feature라는 뜻이지 판매 원인이나 독자 선호를 의미하지 않습니다.

### 실습 22 결과 확인 및 정리

TF-IDF가 높다는 것은 현실 세계에서 절대적으로 중요한 단어라는 뜻이 아니라 **현재 문서 집합과 Vectorizer 설정 안에서 상대적으로 두드러지는 텍스트 특징**이라는 뜻입니다.

## 실습 23. 작은 예제로 TfidfVectorizer 적용하기

작은 세 문장 예제에 TfidfVectorizer를 적용해 Count 행렬과 비교합니다.

### 초보자 상세 설명

작은 예제에서 Count 표와 TF-IDF 표를 나란히 비교합니다. Count는 정수 등장 횟수가 보이고 TF-IDF는 소수 가중치가 나타납니다. 같은 문서×단어 구조라도 셀 값의 의미가 달라진다는 점을 확인합니다.

In [ ]:
# TfidfVectorizer를 불러옵니다.
from sklearn.feature_extraction.text import TfidfVectorizer

# 작은 예제용 Vectorizer를 만듭니다.
tfidf_sample_vectorizer = TfidfVectorizer()

# 단어 사전을 학습하고 TF-IDF 행렬로 변환합니다.
X_tfidf_sample = tfidf_sample_vectorizer.fit_transform(sample_docs)

# 단어 목록을 가져옵니다.
tfidf_sample_terms = tfidf_sample_vectorizer.get_feature_names_out()

# 작은 예제이므로 Dense 배열로 바꿔 DataFrame으로 확인합니다.
tfidf_sample_df = pd.DataFrame(
    X_tfidf_sample.toarray(),
    columns=tfidf_sample_terms,
    index=sample_docs,
)

tfidf_sample_df.round(3)

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# Count와 TF-IDF 작은 예제를 나란히 비교합니다.
print('Count 행렬')
display(sample_count_df)

print('TF-IDF 행렬')
display(tfidf_sample_df.round(3))

print('Count shape:', X_count_sample.shape)
print('TF-IDF shape:', X_tfidf_sample.shape)

### 실습 23 결과 확인 및 정리

Count 행렬은 주로 정수 등장 횟수지만 TF-IDF는 가중치이므로 소수 값이 나옵니다. 여러 문서에 반복되는 단어와 한 문서에만 등장하는 단어의 값이 어떻게 달라지는지 직접 확인합니다.

## 실습 24. TfidfVectorizer가 만든 단어 확인하기

TF-IDF도 CountVectorizer와 마찬가지로 **행 = 문서, 열 = 단어** 구조를 만듭니다. 다만 셀 값이 단순 Count가 아니라 TF-IDF 가중치입니다.

### 초보자 상세 설명

Count와 TF-IDF 모두 행은 문서, 열은 단어입니다. 차이는 셀 값으로 Count는 등장 횟수, TF-IDF는 문서 안 빈도와 전체 문서에서의 흔함을 반영한 가중치입니다.

In [ ]:
# 작은 예제에서 TF-IDF가 만든 feature 목록을 확인합니다.
print(tfidf_sample_vectorizer.get_feature_names_out())

# Count와 TF-IDF 표를 나란히 확인합니다.
print("Count")
display(sample_count_df)

print("TF-IDF")
display(tfidf_sample_df.round(3))

### 실습 24 결과 확인 및 정리

두 표의 구조는 비슷하지만 셀 값의 의미가 다릅니다. Count는 등장 횟수이고 TF-IDF는 문서 안 빈도와 전체 문서에서의 흔함을 함께 반영한 가중치입니다.

## 실습 25. 단어별 IDF 값 확인하기

TfidfVectorizer가 학습한 각 단어의 IDF 값을 직접 확인합니다.

### 초보자 상세 설명

idf_를 직접 보면 여러 문서에 널리 등장하는 단어와 일부 문서에만 등장하는 단어의 IDF가 어떻게 다른지 확인할 수 있습니다. 절대값을 외우기보다 DF와 함께 상대적인 차이를 이해합니다.

In [ ]:
# 각 feature와 IDF 값을 표로 만듭니다.
idf_df = pd.DataFrame({
    "단어": tfidf_sample_vectorizer.get_feature_names_out(),
    "IDF": tfidf_sample_vectorizer.idf_,
})

# IDF가 높은 순서로 확인합니다.
idf_df.sort_values("IDF", ascending=False)

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 작은 예제에서 각 단어의 DF와 IDF를 함께 봅니다.
sample_df_values = np.asarray((X_count_sample > 0).sum(axis=0)).ravel()

idf_compare = pd.DataFrame({
    '단어': tfidf_sample_vectorizer.get_feature_names_out(),
    '문서빈도_DF': sample_df_values,
    'IDF': tfidf_sample_vectorizer.idf_,
})

idf_compare.sort_values(['문서빈도_DF', 'IDF'])

### 실습 25 결과 확인 및 정리

IDF 숫자 자체를 외울 필요는 없습니다. 어떤 단어가 여러 문서에 널리 등장하는지 확인하고 그에 따라 IDF가 상대적으로 어떻게 달라지는지 보는 것이 핵심입니다.

## 실습 26. 실제 도서 제목을 TF-IDF로 변환하기

이제 실제 `titles` 전체를 TfidfVectorizer로 변환합니다. 이번 Chapter에서는 **개념 학습과 탐색 목적**으로 전체 데이터에 fit합니다.

### 초보자 상세 설명

실제 titles 전체에 TF-IDF를 적용한 뒤 Count와 문서 수, feature 수를 비교합니다. 같은 제목과 같은 기본 token 설정이라면 구조가 같거나 매우 비슷해야 하며, 다르다면 옵션이나 전처리 조건이 달라졌는지 확인합니다.

In [ ]:
# 실제 도서 제목용 TfidfVectorizer를 만듭니다.
tfidf_vectorizer = TfidfVectorizer()

# 전체 제목에서 단어 사전과 IDF를 학습하고 TF-IDF 행렬로 변환합니다.
X_tfidf = tfidf_vectorizer.fit_transform(titles)

# feature 단어 목록을 가져옵니다.
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF 행렬 크기:", X_tfidf.shape)
print("문서 수:", X_tfidf.shape[0])
print("단어 수:", X_tfidf.shape[1])

# Count 행렬과 shape도 비교합니다.
print("Count shape:", X_count.shape)
print("TF-IDF shape:", X_tfidf.shape)

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# Count와 TF-IDF가 같은 제목 집합을 사용하는지 확인합니다.
print('제목 수:', len(titles))
print('Count 문서 수:', X_count.shape[0])
print('TF-IDF 문서 수:', X_tfidf.shape[0])
print('feature 집합 동일:', set(count_terms) == set(tfidf_terms))

### 실습 26 결과 확인 및 정리

같은 제목과 같은 기본 토큰 조건을 사용했다면 Count와 TF-IDF의 행·열 수가 같거나 매우 비슷하게 나올 수 있습니다. 직접 실행 결과로 확인합니다.

## 실습 27. 첫 번째 도서의 TF-IDF 주요 단어 확인하기

전체 행렬을 Dense로 바꾸지 않고 첫 번째 제목 행의 0이 아닌 TF-IDF 값만 확인합니다.

### 초보자 상세 설명

첫 번째 제목에서 0이 아닌 TF-IDF 값만 가져와 높은 순서로 정렬합니다. 원래 제목과 나란히 보면서 실제 제목 속 단어인지, 어떤 단어가 상대적으로 높은 가중치를 받았는지 직접 확인합니다.

In [ ]:
# 원래 첫 번째 제목을 확인합니다.
print("도서 제목:", titles.iloc[0])

# 첫 번째 TF-IDF 행을 가져옵니다.
first_tfidf_row = X_tfidf.getrow(0)

# 0이 아닌 feature와 TF-IDF 값을 DataFrame으로 만듭니다.
first_tfidf_df = pd.DataFrame({
    "단어": tfidf_terms[first_tfidf_row.indices],
    "TF-IDF": first_tfidf_row.data,
})

# 큰 값부터 정렬합니다.
first_tfidf_df = first_tfidf_df.sort_values(
    "TF-IDF",
    ascending=False,
).reset_index(drop=True)

first_tfidf_df

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 첫 번째 제목의 0이 아닌 feature 수와 상위 값을 확인합니다.
print('첫 번째 제목:', titles.iloc[0])
print('0이 아닌 feature 수:', first_tfidf_row.nnz)
display(first_tfidf_df.head(10))

### 실습 27 결과 확인 및 정리

원래 제목에 있는 단어만 나오는지, 가장 높은 단어가 무엇인지, 매우 흔한 단어는 상대적으로 값이 낮아지는지 확인합니다.

## 실습 28. 여러 도서의 주요 TF-IDF 단어 확인 함수 만들기

여러 제목을 반복해서 확인할 수 있도록 작은 함수를 만듭니다.

### 초보자 상세 설명

show_top_tfidf_terms 함수는 같은 확인 작업을 여러 문서에서 반복하기 위한 함수입니다. 함수가 반환한 주요 단어는 항상 원래 제목과 함께 읽고, doc_index가 데이터 범위를 넘지 않게 주의합니다.

In [ ]:
def show_top_tfidf_terms(doc_index, top_n=5):
    # 지정한 문서 한 행만 가져옵니다.
    row = X_tfidf.getrow(doc_index)

    # 0이 아닌 단어와 TF-IDF 값을 표로 만듭니다.
    result = pd.DataFrame({
        "단어": tfidf_terms[row.indices],
        "TF-IDF": row.data,
    })

    # TF-IDF가 높은 단어를 top_n개만 남깁니다.
    result = result.sort_values(
        "TF-IDF",
        ascending=False,
    ).head(top_n)

    print("도서 제목:", titles.iloc[doc_index])
    return result.reset_index(drop=True)

# 몇 개 제목을 직접 확인합니다.
display(show_top_tfidf_terms(0, top_n=5))

if len(titles) > 10:
    display(show_top_tfidf_terms(10, top_n=5))

if len(titles) > 100:
    display(show_top_tfidf_terms(100, top_n=5))

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 여러 제목에 같은 함수를 적용해 결과를 비교합니다.
check_indices = [0, 1, 2, 10]

for idx in check_indices:
    if idx < len(titles):
        print('=' * 60)
        display(show_top_tfidf_terms(idx, top_n=5))

### 실습 28 결과 확인 및 정리

함수의 `doc_index`는 제목의 행 번호입니다. 데이터 길이보다 큰 번호를 넣지 않도록 조건을 넣었습니다.

## 실습 29. 전체 데이터에서 평균 TF-IDF가 높은 단어 확인하기

각 단어의 TF-IDF 값을 전체 문서에서 평균내어 탐색합니다.

### 초보자 상세 설명

각 단어의 TF-IDF를 전체 문서에서 평균내면 전체 데이터에서 평균적으로 값이 큰 feature를 탐색할 수 있습니다. 다만 평균 TF-IDF 상위 단어를 베스트셀러의 원인으로 해석하지 않습니다.

In [ ]:
# 각 열(feature)의 평균 TF-IDF를 계산합니다.
mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()

# 단어와 평균 TF-IDF를 DataFrame으로 만듭니다.
tfidf_summary = pd.DataFrame({
    "단어": tfidf_terms,
    "평균_TFIDF": mean_tfidf,
})

# 평균 TF-IDF가 높은 순서로 정렬합니다.
tfidf_summary = tfidf_summary.sort_values(
    "평균_TFIDF",
    ascending=False,
).reset_index(drop=True)

tfidf_summary.head(30)

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 평균 TF-IDF 상위 10개와 값의 범위를 확인합니다.
display(tfidf_summary.head(10))
print('평균 TF-IDF 최소값:', tfidf_summary['평균_TFIDF'].min())
print('평균 TF-IDF 최대값:', tfidf_summary['평균_TFIDF'].max())

### 실습 29 결과 확인 및 정리

평균 TF-IDF가 높다는 이유만으로 그 단어가 베스트셀러의 원인이라고 해석하면 안 됩니다. 현재 문서 집합에서 계산된 **텍스트 feature의 상대적 크기**입니다.

## 실습 30. TF-IDF 상위 단어 저장하기

평균 TF-IDF 기준 상위 30개 단어를 CSV로 저장합니다.

### 초보자 상세 설명

TF-IDF 상위 결과도 CSV로 저장한 뒤 다시 읽어 파일 존재, 행 수, 한글 표시를 확인합니다. 결과물은 생성과 검증을 한 세트로 생각합니다.

In [ ]:
# 평균 TF-IDF 상위 30개를 복사합니다.
tfidf_top30 = tfidf_summary.head(30).copy()

# 저장 경로입니다.
TFIDF_TOP_PATH = "notebooks/book-text-ml/chapter03_tfidf_top_terms.csv"

tfidf_top30.to_csv(
    TFIDF_TOP_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료:", TFIDF_TOP_PATH)

# 다시 읽어 정상 저장 여부를 확인합니다.
pd.read_csv(TFIDF_TOP_PATH, encoding="utf-8-sig").head()

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 저장한 TF-IDF CSV 파일을 다시 확인합니다.
tfidf_path = Path(TFIDF_TOP_PATH)
print('파일 존재:', tfidf_path.exists())

tfidf_saved = pd.read_csv(TFIDF_TOP_PATH, encoding='utf-8-sig')
print('저장된 행 수:', len(tfidf_saved))
display(tfidf_saved.head())

### 실습 30 결과 확인 및 정리

두 번째 필수 결과물인 `chapter03_tfidf_top_terms.csv`를 만들었습니다.

## 실습 31. Count 상위 단어와 TF-IDF 상위 단어 비교하기

Count와 평균 TF-IDF의 상위 결과를 옆으로 놓고 비교합니다.

### 초보자 상세 설명

Count 상위 목록과 TF-IDF 상위 목록을 옆으로 비교하면 두 방식이 어떤 단어를 높게 보는지 확인할 수 있습니다. Count에서 높지만 TF-IDF에서 내려간 단어는 여러 문서에 널리 퍼져 있는지 DF를 함께 확인해 볼 수 있습니다.

In [ ]:
# 두 상위 목록을 같은 행에 나란히 배치합니다.
comparison = pd.DataFrame({
    "Count_상위단어": count_top30["단어"].reset_index(drop=True),
    "Count_전체등장횟수": count_top30["전체등장횟수"].reset_index(drop=True),
    "TFIDF_상위단어": tfidf_top30["단어"].reset_index(drop=True),
    "평균_TFIDF": tfidf_top30["평균_TFIDF"].reset_index(drop=True),
})

comparison.head(20)

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 두 Top30 목록에 동시에 들어 있는 단어를 확인합니다.
count_top_set = set(count_top30['단어'])
tfidf_top_set = set(tfidf_top30['단어'])
common_top_terms = count_top_set & tfidf_top_set

print('공통 단어 수:', len(common_top_terms))
print('공통 단어:', sorted(common_top_terms))

### 실습 31 결과 확인 및 정리

Count에서 높지만 TF-IDF에서는 상대적으로 내려간 단어가 있는지, 두 목록 모두 높은 단어가 있는지, TF-IDF에서 새롭게 눈에 띄는 단어가 있는지 확인합니다. 차이는 **문서 빈도와 가중치 방식의 차이**로 생각해 볼 수 있습니다.

## 실습 32. 특정 단어가 몇 개 문서에 등장하는지 확인하기

TF-IDF 차이를 이해하려면 특정 단어의 DF(Document Frequency)를 직접 확인해 볼 수 있습니다.

### 초보자 상세 설명

DF 함수는 Count 행렬의 특정 단어 열에서 0보다 큰 문서가 몇 개인지 셉니다. 전체 등장 횟수와 문서 빈도를 함께 보면 한 문서에서 반복되는 단어인지 여러 문서에 널리 퍼진 단어인지 구분할 수 있습니다.

In [ ]:
def document_frequency(term):
    # 단어가 CountVectorizer 사전에 없으면 0을 반환합니다.
    if term not in count_vectorizer.vocabulary_:
        return 0

    # 해당 단어의 열 번호를 찾습니다.
    column_index = count_vectorizer.vocabulary_[term]

    # 그 열만 가져옵니다.
    column = X_count[:, column_index]

    # 값이 0보다 큰 문서의 개수를 셉니다.
    return int((column > 0).sum())

print("데이터 DF:", document_frequency("데이터"))
print("파이썬 DF:", document_frequency("파이썬"))

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 한 단어의 전체 등장 횟수와 DF를 함께 계산하는 함수입니다.
def term_statistics(term):
    if term not in count_vectorizer.vocabulary_:
        return {'단어': term, '전체등장횟수': 0, '문서빈도_DF': 0}

    col_idx = count_vectorizer.vocabulary_[term]
    column = X_count[:, col_idx]
    return {
        '단어': term,
        '전체등장횟수': int(column.sum()),
        '문서빈도_DF': int((column > 0).sum()),
    }

display(pd.DataFrame([
    term_statistics('데이터'),
    term_statistics('파이썬'),
]))

### 실습 32 결과 확인 및 정리

DF가 크면 많은 제목에 등장하고, DF가 작으면 적은 제목에 등장합니다. 이 값을 TF-IDF 결과와 함께 비교하면 IDF의 역할을 이해하기 쉬워집니다.

## 실습 33. CountVectorizer와 TfidfVectorizer의 역할 비교

두 Vectorizer를 다음처럼 정리할 수 있습니다.

| 구분 | CountVectorizer | TfidfVectorizer |
|---|---|---|
| 기본 값 | 단어 등장 횟수 | TF-IDF 가중치 |
| 값 형태 | 주로 정수 | 실수 |
| 전체 문서의 흔함 고려 | 직접 고려하지 않음 | IDF로 고려 |
| 장점 | 단순하고 직관적 | 흔한 단어의 영향 일부 조정 |
| 활용 | 빈도 기반 특징 | 분류·유사도 등 텍스트 특징 |

둘 중 하나가 항상 더 좋다고 단정하지 않습니다. 데이터와 분석 목적, 이후 모델 성능을 실제로 비교해 판단해야 합니다.

### 초보자 상세 설명

CountVectorizer와 TfidfVectorizer 중 하나가 항상 더 좋다고 단정하지 않습니다. 둘은 다른 방식으로 텍스트를 숫자로 표현하며 실제 분류나 유사도 작업에서는 같은 train/test 조건에서 모델 성능을 비교해 선택해야 합니다.

### 실습 33 결과 확인 및 정리

Count는 “몇 번 나왔는가”에 직접 초점을 두고, TF-IDF는 “이 문서에서는 보이면서 전체에서는 얼마나 흔한가”까지 함께 반영합니다.

## 실습 34. 같은 단어 사전을 사용해서 Count와 TF-IDF를 비교하려면

같은 제목과 같은 기본 옵션을 사용했다면 두 Vectorizer가 같은 feature 집합을 만드는지 직접 비교할 수 있습니다.

### 초보자 상세 설명

두 Vectorizer의 feature 집합이 같은지 먼저 비교해야 숫자 행렬 차이도 제대로 해석할 수 있습니다. 옵션이 조금이라도 다르면 feature 목록과 순서가 달라질 수 있습니다.

In [ ]:
# Count와 TF-IDF의 feature 집합을 각각 만듭니다.
count_feature_set = set(count_vectorizer.get_feature_names_out())
tfidf_feature_set = set(tfidf_vectorizer.get_feature_names_out())

print("Count 단어 수:", len(count_feature_set))
print("TF-IDF 단어 수:", len(tfidf_feature_set))
print("같은 단어 집합인가?:", count_feature_set == tfidf_feature_set)

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# feature 집합뿐 아니라 순서까지 같은지 확인합니다.
count_features = count_vectorizer.get_feature_names_out()
tfidf_features = tfidf_vectorizer.get_feature_names_out()

print('feature 집합 동일:', set(count_features) == set(tfidf_features))
print('feature 순서까지 동일:', np.array_equal(count_features, tfidf_features))

### 실습 34 결과 확인 및 정리

설정을 바꾸지 않았다면 같은 단어 집합이 만들어질 가능성이 높습니다. 하지만 `stop_words`, `min_df`, tokenizer 등의 옵션이 달라지면 feature 집합도 달라질 수 있습니다.

## 실습 35. stop_words 옵션 이해하기

Vectorizer에서도 불용어를 지정할 수 있습니다. 하지만 AI가 추천한 긴 불용어 목록을 검토 없이 그대로 쓰는 방식은 피합니다.

권장 흐름은 **상위 단어 확인 → 원본 제목 확인 → 의미가 적은지 판단 → 필요한 단어만 추가 → 결과 다시 확인**입니다.

### 초보자 상세 설명

불용어는 결과를 크게 바꿀 수 있으므로 상위 단어를 보기도 전에 긴 목록을 넣지 않습니다. 실제 제목에서 사용 맥락을 확인하고 분석 목적에 도움이 적은 단어만 제한적으로 제외합니다.

In [ ]:
# 예시 불용어입니다. 이번 셀은 옵션 사용법을 확인하기 위한 예제입니다.
stop_words_example = [
    "그리고",
    "대한",
    "위한",
]

vectorizer_with_stopwords = TfidfVectorizer(
    stop_words=stop_words_example,
)

X_stopwords_example = vectorizer_with_stopwords.fit_transform(sample_docs)

print(vectorizer_with_stopwords.get_feature_names_out())

### 실습 35 결과 확인 및 정리

불용어를 추가했다는 사실 자체보다 **왜 그 단어를 제외했는지 설명할 수 있는가**가 중요합니다.

## 실습 36. max_features 옵션 이해하기

`max_features`는 사용할 feature 수의 최대값을 제한하는 옵션입니다. 단어 수가 매우 많을 때 사용할 수 있지만 무조건 1000 같은 숫자를 정답처럼 적용하면 안 됩니다.

### 초보자 상세 설명

max_features는 feature 수의 상한입니다. 계산량을 줄일 수 있지만 중요한 단어가 잘릴 수도 있으므로 적용 전 전체 feature 수와 적용 후 수를 비교합니다.

In [ ]:
# 현재 기본 설정에서 만든 단어 수를 먼저 확인합니다.
print("현재 단어 수:", len(tfidf_terms))

# 예시로 최대 1,000개 feature를 사용하는 Vectorizer를 만듭니다.
limited_vectorizer = TfidfVectorizer(
    max_features=1000,
)

X_limited = limited_vectorizer.fit_transform(titles)

print("max_features 적용 후 단어 수:", X_limited.shape[1])

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# max_features 적용 전후 feature 수를 비교합니다.
print('기본 TF-IDF feature 수:', X_tfidf.shape[1])
print('max_features=1000 feature 수:', X_limited.shape[1])

### 실습 36 결과 확인 및 정리

옵션 적용 전 전체 feature 수를 먼저 확인한 뒤, 왜 제한이 필요한지 판단해야 합니다.

## 실습 37. min_df 옵션 이해하기

`min_df`는 너무 적은 문서에만 등장하는 단어를 제외하는 옵션입니다. `min_df=2`는 대략 **한 문서에만 등장한 단어를 제외하고 2개 이상의 문서에 등장한 단어를 사용**하는 의미입니다.

### 초보자 상세 설명

min_df=2는 한 문서에만 등장한 단어를 제외하고 두 문서 이상에 등장한 단어를 남기는 설정입니다. 드문 전문용어가 중요한 특징일 수 있으므로 무조건 적용하지 않고 전후 결과를 비교합니다.

In [ ]:
# 2개 이상의 문서에 등장한 단어만 사용하는 예제입니다.
vectorizer_min_df = TfidfVectorizer(
    min_df=2,
)

X_min_df = vectorizer_min_df.fit_transform(titles)

print("기본 단어 수:", len(tfidf_terms))
print("min_df=2 단어 수:", X_min_df.shape[1])

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# min_df 적용 전후 feature 수를 비교합니다.
print('기본 feature 수:', X_tfidf.shape[1])
print('min_df=2 feature 수:', X_min_df.shape[1])
print('줄어든 feature 수:', X_tfidf.shape[1] - X_min_df.shape[1])

### 실습 37 결과 확인 및 정리

드문 단어가 항상 쓸모없는 것은 아닙니다. 특정 전문용어가 도서 분야를 구분하는 중요한 특징일 수도 있으므로 적용 전후 feature 수와 이후 모델 성능을 비교해야 합니다.

## 실습 38. max_df 옵션 이해하기

`max_df`는 지나치게 많은 문서에 등장하는 단어를 자동으로 제외할 때 사용할 수 있습니다.

### 초보자 상세 설명

max_df=0.95는 95%를 초과하는 문서에 등장하는 매우 흔한 단어를 제외하는 설정입니다. 실제 데이터에서는 적용해도 feature 수가 변하지 않을 수 있으므로 효과를 숫자로 확인합니다.

In [ ]:
# 전체 문서의 95%를 초과해 등장하는 단어를 제외하는 예제입니다.
vectorizer_max_df = TfidfVectorizer(
    max_df=0.95,
)

X_max_df = vectorizer_max_df.fit_transform(titles)

print("기본 단어 수:", len(tfidf_terms))
print("max_df=0.95 단어 수:", X_max_df.shape[1])

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# max_df 적용 전후 feature 수를 비교합니다.
print('기본 feature 수:', X_tfidf.shape[1])
print('max_df=0.95 feature 수:', X_max_df.shape[1])
print('제외된 feature 수:', X_tfidf.shape[1] - X_max_df.shape[1])

### 실습 38 결과 확인 및 정리

이번 데이터에서는 95% 이상의 제목에 등장하는 단어가 거의 없을 수도 있습니다. 옵션을 넣었다는 사실보다 **왜 넣었고 실제 feature 수가 어떻게 변했는지**가 중요합니다.

## 실습 39. ngram_range 개념 맛보기

기본 Vectorizer는 보통 한 단어씩 feature로 만드는 unigram 방식입니다. `ngram_range=(1, 2)`를 사용하면 한 단어와 두 단어 조합을 함께 사용할 수 있습니다.

### 초보자 상세 설명

ngram_range=(1,2)는 한 단어와 두 단어 조합을 함께 feature로 사용합니다. '데이터', '분석'뿐 아니라 '데이터 분석'도 하나의 feature가 될 수 있으며 feature 수가 크게 늘 수 있습니다.

In [ ]:
# unigram과 bigram을 함께 사용하는 작은 예제입니다.
bigram_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
)

X_bigram = bigram_vectorizer.fit_transform(sample_docs)

print(bigram_vectorizer.get_feature_names_out())

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# unigram과 bigram 포함 설정의 feature 수를 비교합니다.
unigram_vectorizer = TfidfVectorizer()
X_unigram = unigram_vectorizer.fit_transform(sample_docs)

print('unigram feature 수:', X_unigram.shape[1])
print('(1, 2) ngram feature 수:', X_bigram.shape[1])
print(bigram_vectorizer.get_feature_names_out())

### 실습 39 결과 확인 및 정리

`데이터 분석`, `파이썬 데이터` 같은 두 단어 조합도 feature가 될 수 있습니다. 이번 Chapter에서는 확장 개념으로만 확인하고 기본 unigram을 먼저 이해합니다.

## 실습 40. Chapter 02 형태소 분석 결과와 연결하기

Chapter 02에서 사용한 Kiwi 전처리 철학을 Vectorizer 앞단에도 적용할 수 있습니다. 먼저 각 제목을 형태소 분석해 사용할 단어만 남기고 다시 공백으로 연결한 문자열을 만들 수 있습니다.

이 방식이 기본 Vectorizer보다 무조건 더 좋다고 단정하지 않습니다. 이후 모델 성능을 비교해 판단해야 합니다.

### 초보자 상세 설명

Chapter 02의 Kiwi 전처리를 앞단에 적용하면 원하는 품사와 길이 조건을 사용한 뒤 Vectorizer에 넣을 수 있습니다. 하지만 기본 문자열 방식보다 무조건 좋다고 단정하지 않고 feature와 이후 모델 성능을 비교해야 합니다.

In [ ]:
# Chapter 02에서 설치한 Kiwi를 사용합니다.
from kiwipiepy import Kiwi

kiwi = Kiwi()

# 사용할 품사와 예시 불용어입니다.
USE_TAGS = {"NNG", "NNP", "SL"}
STOP_WORDS_KIWI = {"도서", "책"}

def extract_terms(text):
    terms = []

    for token in kiwi.tokenize(str(text)):
        word = token.form.strip().lower()

        # 사용할 품사만 남깁니다.
        if token.tag not in USE_TAGS:
            continue

        # 2글자 미만 단어를 제외합니다.
        if len(word) < 2:
            continue

        # 예시 불용어를 제외합니다.
        if word in STOP_WORDS_KIWI:
            continue

        terms.append(word)

    return terms

# 각 제목을 Kiwi로 정리한 뒤 다시 공백 문자열로 연결합니다.
processed_titles = titles.apply(
    lambda text: " ".join(extract_terms(text))
)

processed_titles.head()

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 원본 제목과 Kiwi 전처리 제목을 나란히 비교합니다.
kiwi_compare = pd.DataFrame({
    '원본 제목': titles.head(10),
    'Kiwi 전처리 제목': processed_titles.head(10),
})
display(kiwi_compare)

# 전처리 결과에도 TF-IDF를 적용해 shape를 비교합니다.
kiwi_tfidf_vectorizer = TfidfVectorizer()
X_kiwi_tfidf = kiwi_tfidf_vectorizer.fit_transform(processed_titles)

print('기본 제목 TF-IDF shape:', X_tfidf.shape)
print('Kiwi 전처리 TF-IDF shape:', X_kiwi_tfidf.shape)

### 실습 40 결과 확인 및 정리

원본 제목을 바로 Vectorizer에 넣는 방식과 Kiwi로 전처리한 제목을 넣는 방식은 서로 다른 feature를 만들 수 있습니다. 어느 방식이 더 좋은지는 다음 Chapter의 실제 분류 성능까지 비교해야 판단할 수 있습니다.

## 실습 41. Custom tokenizer 방식은 언제 사용할까?

Vectorizer에 직접 tokenizer 함수를 전달할 수도 있습니다. 다만 초보자 과정에서는 **전처리된 문자열을 먼저 만들고 중간 결과를 확인한 뒤 Vectorizer에 넣는 방식**이 더 이해하기 쉽습니다.

### 초보자 상세 설명

Vectorizer에 custom tokenizer를 직접 전달할 수도 있지만 초보자에게는 중간 결과가 숨겨져 디버깅이 어려울 수 있습니다. 원본 제목 → Kiwi 결과 → 전처리 문자열 → Vectorizer 순서로 단계별로 확인하는 방식이 더 이해하기 쉽습니다.

In [ ]:
def kiwi_tokenizer(text):
    # 앞에서 만든 extract_terms() 결과를 그대로 반환합니다.
    return extract_terms(text)

# Custom tokenizer를 사용하는 방법의 예입니다.
custom_vectorizer = TfidfVectorizer(
    tokenizer=kiwi_tokenizer,
    token_pattern=None,
    lowercase=False,
)

# 작은 예제로 동작만 확인합니다.
X_custom = custom_vectorizer.fit_transform(sample_docs)

print(custom_vectorizer.get_feature_names_out())

### 실습 41 결과 확인 및 정리

Custom tokenizer는 가능하지만 중간 결과를 숨길 수 있어 초보자에게는 디버깅이 어려울 수 있습니다. **원본 제목 → Kiwi 결과 → 전처리 제목 → Vectorizer** 순서로 나누어 확인하는 방식이 학습에 더 좋습니다.

## 실습 42. Vectorizer가 만든 결과를 시각적으로 이해하기

작은 Count 예제의 표를 다시 생각해 봅니다.

| 문서 | 데이터 | 머신러닝 | 분석 | 입문 | 파이썬 |
|---|---:|---:|---:|---:|---:|
| 파이썬 데이터 분석 | 1 | 0 | 1 | 0 | 1 |
| 파이썬 머신러닝 | 0 | 1 | 0 | 0 | 1 |
| 데이터 분석 입문 | 1 | 0 | 1 | 1 | 0 |

이 표의 **한 행 전체가 문서 하나를 숫자로 표현한 벡터**입니다. TF-IDF에서도 구조는 같고 셀 값만 Count 대신 가중치가 됩니다.

### 초보자 상세 설명

문서 하나는 결국 같은 길이의 숫자 배열 하나가 됩니다. 전체 feature가 5개면 모든 문서는 길이 5의 벡터가 되고, 이 벡터가 이후 분류나 유사도 계산의 입력이 됩니다.

### 실습 42 결과 확인 및 정리

`[1, 0, 1, 0, 1]` 같은 한 줄이 머신러닝 모델에 들어갈 수 있는 숫자 표현입니다. 실제 열 순서는 반드시 Vectorizer가 만든 feature 순서를 기준으로 해석합니다.

## 실습 43. 행렬에서 0이 많다는 의미

각 제목은 짧지만 전체 데이터에는 수많은 단어가 있습니다. 따라서 한 제목에는 전체 feature 중 일부만 등장하고 나머지는 대부분 0이 됩니다.

예를 들어 한 문서의 벡터가 `[0, 0, 0.7, 0, 0.4, 0, ...]`처럼 0이 많은 형태가 됩니다. 이것이 희소 행렬을 사용하는 이유와 연결됩니다.

### 초보자 상세 설명

제목은 짧고 전체 feature는 많기 때문에 한 제목에서 0이 아닌 값은 매우 적습니다. 이것이 텍스트 행렬이 sparse해지는 이유입니다.

### 실습 43 결과 확인 및 정리

텍스트 벡터는 일반적으로 feature 수는 많고 한 문서에서 실제로 사용하는 feature는 적기 때문에 매우 sparse한 구조가 되기 쉽습니다.

## 실습 44. 0이 아닌 값의 개수 확인하기

전체 행렬의 셀 수와 실제로 저장된 0이 아닌 값의 개수를 비교해 희소성을 직접 확인합니다.

### 초보자 상세 설명

nnz는 0이 아닌 값의 개수입니다. 전체 셀 수와 nnz를 비교해 0의 비율을 계산하면 희소 행렬을 사용하는 이유를 실제 숫자로 확인할 수 있습니다.

In [ ]:
# 행과 열 수를 가져옵니다.
rows, cols = X_tfidf.shape

# 전체 셀 수 = 행 × 열
total_cells = rows * cols

# sparse matrix의 nnz는 0이 아닌 값의 개수입니다.
non_zero_cells = X_tfidf.nnz

print("전체 셀 수:", total_cells)
print("0이 아닌 셀 수:", non_zero_cells)

# 전체 셀 중 0의 비율을 계산합니다.
zero_ratio = 1 - (non_zero_cells / total_cells)
print("0의 비율:", round(zero_ratio, 4))

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 0과 0이 아닌 값의 비율이 합쳐서 1이 되는지 확인합니다.
non_zero_ratio = non_zero_cells / total_cells
print('0이 아닌 비율:', round(non_zero_ratio, 4))
print('0의 비율:', round(zero_ratio, 4))
print('합:', round(non_zero_ratio + zero_ratio, 4))

### 실습 44 결과 확인 및 정리

0의 비율이 매우 높다면 전체 Dense 배열로 저장하는 것보다 sparse matrix 구조가 훨씬 효율적이라는 점을 직접 확인할 수 있습니다.

## 실습 45. AI에게 결과 해석을 요청할 때 주의하기

AI에게 단순히 “TF-IDF 결과를 분석해 주세요”라고 숫자 없이 요청하지 않습니다. 실제 실행 결과를 함께 전달해야 합니다.

### AI에게 질문

> 교보문고 베스트셀러 도서 제목을 TfidfVectorizer로 변환했습니다.  
> 실제 결과는 다음과 같습니다.
>
> - 문서 수: [실제 값]
> - 단어 수: [실제 값]
> - 평균 TF-IDF 상위 10개: [실제 출력]
>
> 다음 조건으로 설명해 주세요.
>
> 1. 숫자를 임의로 만들지 말 것
> 2. TF-IDF가 높은 단어를 판매 원인으로 단정하지 말 것
> 3. 현재 문서 집합에서 상대적으로 두드러진 텍스트 특징이라는 수준으로 설명할 것
> 4. 5문장 이내로 작성할 것

### AI 답변 기준

실제 결과만 바탕으로 **관찰 가능한 특징**을 설명하고, TF-IDF가 높은 이유를 판매 원인이나 독자 선호로 확대 해석하지 않습니다.

### 초보자 상세 설명

AI에게 결과 해석을 요청할 때는 실제 출력값을 함께 줍니다. 숫자 없이 막연하게 분석을 요청하면 실제 데이터에 없는 내용을 추측할 위험이 있습니다. 해석 범위도 현재 문서 집합의 상대적 텍스트 특징 수준으로 제한합니다.

In [ ]:
# AI에게 전달할 실제 숫자를 먼저 출력합니다.
print("문서 수:", X_tfidf.shape[0])
print("단어 수:", X_tfidf.shape[1])

print("\n평균 TF-IDF 상위 10개")
display(tfidf_summary.head(10))

### 실습 45 결과 확인 및 정리

AI 설명도 최종 답이 아니라 초안입니다. 실제 Notebook 출력과 문장이 일치하는지 다시 비교해야 합니다.

## 실습 46. Count와 TF-IDF 차이를 Markdown으로 정리하기

Notebook에 Count와 TF-IDF의 차이를 본인의 말로 정리합니다. 아래 문장은 실제 실행값을 임의로 만들지 않고 개념 차이를 설명합니다.

### 초보자 상세 설명

Markdown에는 개념만 반복하기보다 실제 Count와 TF-IDF 결과에서 관찰한 차이를 기록합니다. 다만 아직 실행하지 않은 숫자를 미리 만들어 적지 않습니다.

### 실습 46 결과 확인 및 정리

**CountVectorizer와 TF-IDF 비교**

CountVectorizer는 각 도서 제목에서 단어가 등장한 횟수를 숫자로 표현했습니다. 반면 TF-IDF는 한 제목에서의 등장뿐 아니라 전체 제목에서 해당 단어가 얼마나 흔하게 등장하는지도 함께 반영합니다. 실제 결과에서 Count 상위 단어와 평균 TF-IDF 상위 단어는 일부 차이가 생길 수 있습니다. 따라서 단순 등장 횟수와 문서를 구분하는 데 사용할 상대적 가중치는 같은 개념이 아님을 확인할 수 있습니다.

## 실습 47. 일부 도서 결과를 직접 검증하기

몇 개 제목을 선택해 실제 제목과 TF-IDF 주요 단어를 직접 비교합니다.

### 초보자 상세 설명

일부 제목을 직접 검증하면 전체 상위표만으로는 보이지 않는 tokenization 문제를 찾을 수 있습니다. 원문과 주요 TF-IDF 단어를 나란히 보고 이상한 숫자 token이나 지나치게 일반적인 단어가 있는지 확인합니다.

In [ ]:
# 확인할 행 번호 예시입니다.
sample_indices = [0, 10, 20]

for index in sample_indices:
    # 데이터 범위 안의 인덱스만 실행합니다.
    if index < len(titles):
        print("=" * 60)
        print("도서 제목:", titles.iloc[index])
        display(show_top_tfidf_terms(index, top_n=5))

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 원문과 주요 TF-IDF 단어를 같은 흐름에서 확인합니다.
for index in sample_indices:
    if index < len(titles):
        result = show_top_tfidf_terms(index, top_n=5)
        print('=' * 60)
        print('원문:', titles.iloc[index])
        display(result)

### 실습 47 결과 확인 및 정리

상위 단어가 실제 제목에 존재하는지, 숫자나 기호가 이상하게 feature가 되지는 않았는지, 지나치게 일반적인 단어가 높은 값으로 나오지는 않았는지 확인합니다.

## 실습 48. 결과가 이상할 때 확인할 순서

### AI에게 질문

> 다음은 TfidfVectorizer 실행 결과입니다.
>
> 도서 제목: [실제 제목]  
> 상위 TF-IDF 단어: [실제 결과]
>
> 1. 각 단어가 원래 제목에 실제로 존재하는지 확인해야 할 항목을 알려 주세요.
> 2. 결과가 이상할 때 토큰화, 불용어, min_df 등의 관점에서 점검 순서를 알려 주세요.
> 3. 결과에 없는 숫자를 새로 만들지 마세요.

### AI 답변

결과가 이상하다고 바로 전체 코드를 다시 작성하지 않고 다음 순서로 확인합니다.

1. 원본 제목 확인
2. 결측치 처리 확인
3. Vectorizer feature 목록 확인
4. 토큰화 기준 확인
5. 불용어 설정 확인
6. `min_df`, `max_df` 등 옵션 확인
7. 형태소 분석 적용 여부 확인
8. 해당 행의 0이 아닌 값 직접 확인

이 순서로 보면 데이터 문제인지, 토큰화 문제인지, Vectorizer 옵션 문제인지 범위를 좁힐 수 있습니다.

### 초보자 상세 설명

결과가 이상할 때는 전체 코드를 다시 쓰기보다 원문 → 결측치 → feature → token 기준 → 불용어/옵션 → 특정 행 값 순서로 문제 범위를 좁힙니다.

### 실습 48 결과 확인 및 정리

오류나 이상한 결과를 AI에게 질문할 때는 **실제 제목, 실제 출력, 실제 오류 메시지**를 함께 전달하는 것이 가장 중요합니다.

## 실습 49. 학습용 행렬 저장하기

희소 행렬 자체도 선택적으로 저장할 수 있습니다. 이 파일은 이번 Chapter에서 벡터화 결과를 학습하고 확인하기 위한 **학습용 결과물**입니다.

### 초보자 상세 설명

희소 행렬 npz 파일은 이번 Chapter의 탐색용 결과입니다. 다음 Chapter의 모델 평가에 전체 데이터로 fit한 행렬을 그대로 사용하면 테스트 데이터 정보가 전처리 기준에 들어갈 수 있습니다.

In [ ]:
# scipy의 희소 행렬 저장 기능을 불러옵니다.
from scipy.sparse import save_npz

COUNT_MATRIX_PATH = "notebooks/book-text-ml/chapter03_count_matrix.npz"
TFIDF_MATRIX_PATH = "notebooks/book-text-ml/chapter03_tfidf_matrix.npz"

# Count와 TF-IDF 희소 행렬을 저장합니다.
save_npz(COUNT_MATRIX_PATH, X_count)
save_npz(TFIDF_MATRIX_PATH, X_tfidf)

print("Count 행렬 저장:", COUNT_MATRIX_PATH)
print("TF-IDF 행렬 저장:", TFIDF_MATRIX_PATH)

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# npz 파일이 실제로 만들어졌는지 확인합니다.
print('Count matrix 파일 존재:', Path(COUNT_MATRIX_PATH).exists())
print('TF-IDF matrix 파일 존재:', Path(TFIDF_MATRIX_PATH).exists())

### 실습 49 결과 확인 및 정리

이 행렬을 다음 Chapter의 모델 평가에 그대로 가져다 쓰면 안 됩니다. Chapter 04에서는 train/test를 먼저 나눈 뒤 **train 데이터로 Vectorizer를 다시 fit**해야 합니다.

## 실습 50. 왜 Chapter 04에서는 전체 데이터에 먼저 fit하면 안 될까?

### AI에게 질문

> 텍스트 분류를 처음 배우고 있습니다.  
> 왜 전체 데이터에 `TfidfVectorizer.fit_transform()`을 먼저 한 뒤 train/test로 나누면 데이터 누수가 될 수 있는지 설명해 주세요.  
> 그리고 `train_test_split` 이후 train에는 `fit_transform()`, test에는 `transform()`만 사용하는 이유를 초보자가 이해할 수 있는 예로 설명해 주세요.

### AI 답변

`fit()`은 단순 변환이 아니라 데이터에서 **단어 사전과 IDF 같은 기준을 학습**합니다.

만약 전체 데이터에 먼저 `fit_transform()`을 하면 Vectorizer가 테스트 데이터에 어떤 단어가 있고, 전체 문서에서 그 단어가 얼마나 흔한지까지 미리 본 상태가 됩니다. 이후 train/test를 나눠도 테스트 데이터 정보가 이미 전처리 기준에 들어간 셈입니다. 이를 **데이터 누수(Data Leakage)**라고 볼 수 있습니다.

시험 문제로 비유하면 테스트 세트를 평가에 쓰기 전에 시험 문제를 보고 단어 사전과 중요도 기준을 만든 것과 비슷합니다. 모델 평가에서는 테스트 데이터가 학습 과정에 영향을 주지 않아야 합니다.

### 초보자 상세 설명

Vectorizer의 fit도 학습 과정입니다. 전체 데이터에 먼저 fit하면 단어 사전과 IDF를 만드는 과정에서 테스트 데이터 정보를 미리 보게 됩니다. 이것이 데이터 누수 문제로 연결됩니다.

### 실습 50 결과 확인 및 정리

이번 Chapter에서 전체 데이터에 `fit_transform()`을 한 것은 **Vectorizer의 구조를 탐색하고 배우기 위한 목적**입니다. 실제 모델 평가에서는 순서가 달라집니다.

## 실습 51. Chapter 04에서 사용할 올바른 순서 미리 보기

Chapter 04에서는 다음 순서를 사용합니다.

**원본 데이터 → X와 y 준비 → train/test 분리 → TF-IDF를 train에 fit → train transform → test는 transform만 → 모델 학습 → test 예측 및 평가**

### 초보자 상세 설명

올바른 순서는 train에서 기준을 학습하고 test에는 그 기준을 적용만 하는 것입니다. train에는 fit_transform, test에는 transform만 사용합니다.

In [ ]:
# Chapter 04에서 사용할 개념적인 흐름의 예시입니다.
# 실제 변수는 Chapter 04에서 train_test_split 이후 만들어집니다.

# X_train_tfidf = vectorizer.fit_transform(X_train)
# X_test_tfidf = vectorizer.transform(X_test)

# 테스트 데이터에 아래처럼 fit_transform()을 다시 사용하면 안 됩니다.
# X_test_tfidf = vectorizer.fit_transform(X_test)

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 작은 예제로 train에는 fit_transform, test에는 transform만 적용합니다.
demo_vectorizer = TfidfVectorizer()
demo_train = sample_docs[:2]
demo_test = sample_docs[2:]

X_demo_train = demo_vectorizer.fit_transform(demo_train)
X_demo_test = demo_vectorizer.transform(demo_test)

print('train shape:', X_demo_train.shape)
print('test shape:', X_demo_test.shape)
print('train에서 학습한 feature:', demo_vectorizer.get_feature_names_out())

### 실습 51 결과 확인 및 정리

테스트 데이터에는 `fit_transform()`을 사용하지 않고 **train에서 이미 학습한 기준으로 transform만 수행**하는 것이 핵심입니다.

## 실습 52. fit과 transform을 구분해서 설명해 보기

이번 Chapter를 마치기 전에 세 용어를 본인의 말로 설명할 수 있어야 합니다.

- **fit** → 데이터에서 단어 사전과 필요한 통계 정보를 학습
- **transform** → 이미 학습한 기준으로 데이터를 숫자로 변환
- **fit_transform** → fit과 transform을 한 번에 수행

CountVectorizer와 TfidfVectorizer 모두 이 구분이 중요합니다.

### 초보자 상세 설명

fit은 기준 학습, transform은 학습된 기준으로 변환, fit_transform은 둘을 한 번에 수행합니다. 이 구분은 scikit-learn 전반에서 반복되며 모델 평가의 기본 원칙입니다.

### 실습 52 결과 확인 및 정리

특히 머신러닝 평가에서는 **어떤 데이터에 fit했는가**가 매우 중요합니다. 테스트 데이터가 fit 과정에 들어가지 않도록 해야 합니다.

## 실습 53. Vectorizer 옵션을 기록하기

재현 가능한 분석을 위해 이번 실습에서 사용한 설정을 기록합니다.

### Chapter 03 벡터화 조건

- 데이터: `book_bestseller_clean.csv`
- 분석 컬럼: `상품명`
- CountVectorizer: 기본 설정
- TfidfVectorizer: 기본 설정
- 실습 목적: 전체 문서에서 벡터화 구조와 feature 확인
- 모델 평가용 Vectorizer: Chapter 04에서 train 데이터 기준으로 별도 fit 예정
- 확장 실습: stop_words, max_features, min_df, max_df, ngram_range, Kiwi 전처리 방식 확인

### 초보자 상세 설명

재현 가능한 분석을 위해 Vectorizer 설정을 Markdown으로 기록합니다. 기본 설정인지, 불용어·min_df·Kiwi 등을 적용했는지를 남겨야 나중에 결과 차이의 원인을 찾기 쉽습니다.

### 실습 53 결과 확인 및 정리

설정을 기록해 두면 나중에 결과가 달라졌을 때 **데이터가 달라진 것인지, Vectorizer 옵션이 달라진 것인지** 확인하기 쉽습니다.

## 실습 54. 재현 가능한 코드 구조로 정리하기

앞에서 각 단계를 이해한 뒤 핵심 흐름만 한 번에 다시 정리합니다. 처음부터 이 셀만 복사하는 정답지가 아니라 **전체 과정을 복습하는 요약 코드**입니다.

### 초보자 상세 설명

마지막 정리 코드는 앞에서 배운 데이터 로딩 → Count → TF-IDF → 요약표 흐름을 한 번에 재현하는 복습용입니다. 새 개념을 추가하는 셀이 아니라 앞의 결과와 같은지 검증하는 용도입니다.

In [ ]:
# 핵심 라이브러리
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 1. 데이터 불러오기
DATA_PATH = "notebooks/book-text-ml/book_bestseller_clean.csv"
df_books_check = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

# 2. 제목 준비
titles_check = (
    df_books_check["상품명"]
    .fillna("")
    .astype(str)
    .str.strip()
)
titles_check = titles_check[titles_check != ""].reset_index(drop=True)

# 3. CountVectorizer
count_vectorizer_check = CountVectorizer()
X_count_check = count_vectorizer_check.fit_transform(titles_check)
count_terms_check = count_vectorizer_check.get_feature_names_out()

# 4. Count 전체 빈도
count_sums_check = np.asarray(X_count_check.sum(axis=0)).ravel()
count_summary_check = pd.DataFrame({
    "단어": count_terms_check,
    "전체등장횟수": count_sums_check,
}).sort_values(
    "전체등장횟수",
    ascending=False,
).reset_index(drop=True)

# 5. TF-IDF
tfidf_vectorizer_check = TfidfVectorizer()
X_tfidf_check = tfidf_vectorizer_check.fit_transform(titles_check)
tfidf_terms_check = tfidf_vectorizer_check.get_feature_names_out()

# 6. 평균 TF-IDF
mean_tfidf_check = np.asarray(X_tfidf_check.mean(axis=0)).ravel()
tfidf_summary_check = pd.DataFrame({
    "단어": tfidf_terms_check,
    "평균_TFIDF": mean_tfidf_check,
}).sort_values(
    "평균_TFIDF",
    ascending=False,
).reset_index(drop=True)

# 7. 핵심 결과 확인
print("Count shape:", X_count_check.shape)
print("TF-IDF shape:", X_tfidf_check.shape)

display(count_summary_check.head(30))
display(tfidf_summary_check.head(30))

### 추가로 직접 확인하기

위 코드가 오류 없이 실행됐다는 사실만 확인하지 않고, 아래 코드로 **숫자와 구조가 앞의 설명과 실제로 맞는지** 직접 확인합니다. 수업자료가 강조하는 '실행 → 결과 확인 → 검증' 단계입니다.

In [ ]:
# 마지막 요약 코드 결과가 앞의 결과와 같은지 확인합니다.
print('Count shape 동일:', X_count_check.shape == X_count.shape)
print('TF-IDF shape 동일:', X_tfidf_check.shape == X_tfidf.shape)

print('Count Top10 동일:',
      count_summary_check.head(10)['단어'].tolist() ==
      count_summary.head(10)['단어'].tolist())

print('TF-IDF Top10 동일:',
      tfidf_summary_check.head(10)['단어'].tolist() ==
      tfidf_summary.head(10)['단어'].tolist())

### 실습 54 결과 확인 및 정리

이 셀의 목적은 이번 Chapter에서 배운 전체 흐름을 한 번에 복습하는 것입니다. 각 단계의 의미를 이해하지 않은 채 이 코드만 복사해서 사용하는 것은 피합니다.

## 실습 55. Notebook 최종 실행 확인

제출 또는 다음 Chapter로 넘어가기 전에 **Kernel Restart → Run All**로 처음부터 마지막까지 다시 실행합니다.

### 확인 체크리스트

- [ ] `book_bestseller_clean.csv`를 정상적으로 불러왔다.
- [ ] 상품명 문자열을 정리했다.
- [ ] 작은 예제로 Bag of Words를 설명할 수 있다.
- [ ] CountVectorizer의 `fit_transform()`을 실행했다.
- [ ] `get_feature_names_out()` 결과를 확인했다.
- [ ] 행이 문서이고 열이 단어라는 것을 이해했다.
- [ ] Count 행렬의 값이 단어 등장 횟수임을 이해했다.
- [ ] 실제 도서 제목 한 행을 직접 검증했다.
- [ ] sparse matrix 전체를 무조건 Dense 배열로 바꾸지 않았다.
- [ ] Count 상위 단어를 확인하고 CSV를 저장했다.
- [ ] TF, DF, IDF의 역할을 구분할 수 있다.
- [ ] TfidfVectorizer를 적용했다.
- [ ] 단어별 IDF 값을 확인했다.
- [ ] 특정 도서의 주요 TF-IDF 단어를 확인했다.
- [ ] 평균 TF-IDF 상위 단어를 확인하고 CSV를 저장했다.
- [ ] Count와 TF-IDF 결과를 비교했다.
- [ ] 결과를 원본 제목과 비교했다.
- [ ] 데이터 누수를 피하기 위해 Chapter 04에서 train에만 fit해야 한다는 점을 이해했다.
- [ ] 실제 실행 결과를 바탕으로 Markdown을 작성했다.

### 초보자 상세 설명

Notebook은 중간에 남아 있는 변수 때문에 일부 셀만 실행하면 우연히 돌아갈 수 있습니다. Kernel Restart 후 Run All을 해서 처음부터 끝까지 순서대로 오류 없이 재현되는지 확인합니다.

### 실습 55 결과 확인 및 정리

마지막 셀까지 순서대로 오류 없이 실행되는지 확인해야 합니다. 중간 셀을 건너뛴 상태에서 우연히 실행되는 Notebook은 재현성이 떨어집니다.

## 실습 56. 이번 Chapter에서 꼭 기억할 개념

1. **텍스트는 머신러닝을 위해 숫자로 변환해야 합니다.**  
   문자열 → feature → 숫자 벡터

2. **Bag of Words는 단어의 등장에 초점을 둡니다.**  
   단어 순서와 깊은 문맥보다 어떤 단어가 몇 번 등장했는지를 사용합니다.

3. **CountVectorizer의 값은 등장 횟수입니다.**  
   행 = 문서, 열 = 단어, 값 = 해당 문서에서 그 단어가 나온 횟수

4. **TF-IDF는 전체 문서에서의 흔함도 함께 고려합니다.**  
   문서 안의 빈도 + 전체 문서에서의 희소성 → TF-IDF 가중치

5. **높은 TF-IDF는 현실 세계의 중요도를 의미하지 않습니다.**  
   현재 문서 집합에서 상대적으로 두드러진 텍스트 특징입니다.

6. **fit과 transform을 구분해야 합니다.**  
   fit = 기준 학습, transform = 학습된 기준으로 변환

7. **모델 평가에서는 테스트 데이터 정보가 학습에 들어가면 안 됩니다.**  
   이번 Chapter의 전체 데이터 `fit_transform()`은 개념 학습용이고 Chapter 04에서는 train/test를 먼저 나눈 뒤 train에만 fit합니다.

### 초보자 상세 설명

이번 Chapter의 핵심은 텍스트를 feature 기반 숫자 벡터로 바꾸는 것입니다. Count는 등장 횟수, TF-IDF는 전체 문서에서의 흔함까지 고려한 가중치이며 Vectorizer도 fit을 수행하는 학습 단계입니다.

### 실습 56 결과 확인 및 정리

이번 Chapter의 가장 중요한 연결점은 **Vectorizer도 학습 과정의 일부**라는 점입니다.

## 실습 57. Chapter 03 결과물 정리

이번 Chapter가 끝나면 최소한 다음 결과물이 있어야 합니다.

- `chapter03.ipynb`
- `chapter03_count_top_terms.csv`
- `chapter03_tfidf_top_terms.csv`

선택적으로 다음 파일도 저장합니다.

- `chapter03_count_matrix.npz`
- `chapter03_tfidf_matrix.npz`

Notebook에는 데이터 불러오기, 상품명 정리, 작은 Bag of Words 예제, CountVectorizer, 단어 사전, 단어-문서 행렬, 희소 행렬, 실제 제목 Count 벡터, Count 상위 단어, TF/DF/IDF, TfidfVectorizer, 단어별 IDF, 도서별 TF-IDF 주요 단어, 평균 TF-IDF 상위 단어, Count와 TF-IDF 비교, 원본 제목 검증, fit/transform 구분, 데이터 누수 주의, 결과 해석 Markdown이 남아 있어야 합니다.

### 초보자 상세 설명

최종 Notebook에는 결과만 남기는 것이 아니라 숫자가 무엇을 뜻하는지 확인한 과정, 원문 검증, 옵션 기록, 데이터 누수 주의, 실제 결과를 사용한 Markdown까지 함께 남아 있어야 합니다.

### 실습 57 결과 확인 및 정리

다음 Chapter에서는 `상품명`을 입력값 X, `분야`를 정답 y로 사용해 실제 분류 모델을 만듭니다. **텍스트를 숫자로 바꾸는 Vectorizer도 학습 과정의 일부이므로 모델 평가에서는 훈련 데이터에만 fit해야 합니다.**

## Chapter 03 실제 결과 해석 Markdown

아래 셀은 임의의 숫자를 쓰지 않고, 이 Notebook에서 실제로 계산한 결과를 가져와 Markdown을 만듭니다.

In [ ]:
# Notebook에 Markdown을 표시하기 위한 기능입니다.
from IPython.display import display, Markdown

# 실제 실행 결과에서 Count 상위 3개를 가져옵니다.
count_word1 = count_top30.iloc[0]["단어"]
count_value1 = count_top30.iloc[0]["전체등장횟수"]
count_word2 = count_top30.iloc[1]["단어"]
count_value2 = count_top30.iloc[1]["전체등장횟수"]
count_word3 = count_top30.iloc[2]["단어"]
count_value3 = count_top30.iloc[2]["전체등장횟수"]

# 실제 실행 결과에서 평균 TF-IDF 상위 3개를 가져옵니다.
tfidf_word1 = tfidf_top30.iloc[0]["단어"]
tfidf_value1 = tfidf_top30.iloc[0]["평균_TFIDF"]
tfidf_word2 = tfidf_top30.iloc[1]["단어"]
tfidf_value2 = tfidf_top30.iloc[1]["평균_TFIDF"]
tfidf_word3 = tfidf_top30.iloc[2]["단어"]
tfidf_value3 = tfidf_top30.iloc[2]["평균_TFIDF"]

chapter03_md = f"""
## Chapter 03 결과 해석

교보문고 베스트셀러 도서 제목을 CountVectorizer와 TfidfVectorizer를 이용해 숫자 벡터로 변환했습니다.

CountVectorizer 결과에서 각 행은 하나의 도서 제목, 각 열은 하나의 단어이며 값은 해당 제목에서 단어가 등장한 횟수를 의미합니다.
실제 Count 기준 상위 3개 단어는 **{count_word1}({count_value1}회)**, **{count_word2}({count_value2}회)**, **{count_word3}({count_value3}회)** 입니다.

TF-IDF에서는 한 제목에서의 단어 등장뿐 아니라 전체 제목에서 해당 단어가 얼마나 흔하게 등장하는지도 함께 반영합니다.
평균 TF-IDF 기준 상위 3개 단어는 **{tfidf_word1}({tfidf_value1:.4f})**, **{tfidf_word2}({tfidf_value2:.4f})**, **{tfidf_word3}({tfidf_value3:.4f})** 입니다.

Count 기준과 평균 TF-IDF 기준의 순위에는 차이가 생길 수 있으며, 이를 통해 단순히 많이 등장하는 단어와 현재 문서 집합에서 상대적으로 두드러지는 특징은 같은 개념이 아님을 확인할 수 있습니다.

이 결과는 현재 도서 제목 데이터와 현재 Vectorizer 설정을 기준으로 한 텍스트 특징이며, 단어의 높은 빈도나 TF-IDF를 도서 판매의 원인 또는 독자 선호로 직접 해석하지 않습니다.
"""

display(Markdown(chapter03_md))

### 최종 정리

위 Markdown은 **실제 실행 결과에서 값을 읽어 자동으로 작성**합니다. 따라서 숫자를 추측하지 않습니다.

Notebook을 제출하기 전에 처음부터 다시 실행해 결과가 정상적으로 표시되는지 확인합니다.

### 자주 발생하는 오류

- **No module named sklearn**  
  `python -m pip install scikit-learn`로 현재 Notebook이 사용하는 Python 환경에 설치합니다.

- **empty vocabulary**  
  제목이 비어 있지 않은지, 불용어나 `min_df` 조건을 너무 강하게 적용하지 않았는지 확인합니다.

- **한 글자 단어가 보이지 않음**  
  기본 CountVectorizer/TfidfVectorizer 토큰 패턴에서 한 글자 토큰이 제외될 수 있습니다.

- **전체 행렬을 출력했더니 너무 큼**  
  전체 sparse matrix를 무조건 `toarray()`로 바꾸지 말고 특정 행만 확인합니다.

- **Count와 TF-IDF 상위 단어가 다름**  
  정상적인 현상일 수 있습니다. 두 방식이 계산하는 값의 의미가 다릅니다.

- **Chapter 02와 단어 목록이 다름**  
  Chapter 02는 Kiwi와 품사 필터를 사용했고 기본 Vectorizer는 다른 토큰 기준을 사용합니다. 분석 조건을 맞춘 뒤 비교합니다.